# Etapa 2 · Parte 2 — Dados ausentes e codificação categórica

**Responsável:** Samuel Ferreira Alves Vieira

Compara (1) imputação categórica por **moda** × categoria explícita **`ausente`** e (2) as codificações
**one-hot**, **one-hot com categorias raras agrupadas**, **ordinal**, **frequência** e **híbrida**
(one-hot em `protocol_type`/`flag`, frequência em `service`).

Protocolo comum (`docs/etapa2_divisao.md`): só a base de treino; `StratifiedGroupKFold` com 5 dobras, sem
embaralhar, com assinaturas idênticas na mesma dobra; Regressão Logística de referência; mediana +
indicador e `StandardScaler` nos numéricos; sem balanceamento e sem redução. Cada variante muda **apenas**
a parte categórica, e toda imputação/codificação é ajustada dentro da dobra de treino.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.encoding import (
    ENCODING_VARIANTS, encoded_dimensions, evaluate_encoding_strategies, unseen_categories_by_fold,
)
from src.preprocessing_stage2 import KNOWN_CATEGORICAL, prepare_modeling_data, report_as_dict

RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_columns', 100)

## 1. Carregamento e regras determinísticas da Etapa 1

O teste sem rótulo **não** participa de nenhum ajuste.

In [ ]:
data_candidates = [
    Path(os.environ['TPIA_TREINO']) if 'TPIA_TREINO' in os.environ else None,
    PROJECT_ROOT / 'data' / 'treino.csv',
    PROJECT_ROOT / 'data' / 'base_nsl_kdd.csv',
    PROJECT_ROOT / 'base_nsl_kdd.csv',
]
DATA_PATH = next((path for path in data_candidates if path is not None and path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Coloque treino.csv (Canvas) em data/.')

raw_data = pd.read_csv(DATA_PATH, sep=None, engine='python')
X, y, preparation_report = prepare_modeling_data(raw_data)
print('Arquivo:', DATA_PATH, '| bruto:', raw_data.shape, '| após regras:', X.shape)
display(pd.Series(report_as_dict(preparation_report), name='resultado').to_frame())

## 2. Diagnóstico dos atributos categóricos

Cardinalidade, ausências e concentração das categorias, com destaque para `service`.

In [ ]:
categorical = list(preparation_report.categorical_columns)
diagnostico = pd.DataFrame({
    'cardinalidade': X[categorical].nunique(),
    'ausentes': X[categorical].isna().sum(),
    'ausentes_%': (X[categorical].isna().mean() * 100).round(2),
    'categorias_<0,1%': [int((X[c].value_counts(normalize=True) < 0.001).sum()) for c in categorical],
})
display(diagnostico)

if 'service' in X.columns:
    service = pd.crosstab(X['service'], y)
    service['total'] = service.sum(axis=1)
    service['anomaly_%'] = (service.get('anomaly', 0) / service['total'] * 100).round(1)
    display(service.sort_values('total', ascending=False).head(15))

## 3. Categorias desconhecidas

Medido entre as dobras (validação × treino da dobra), simulando o que acontece com o teste. Como verificação
descritiva, contamos também os níveis do teste ausentes no treino; essa contagem **não** é usada para ajustar
nada, serve só para confirmar que o codificador precisa de `handle_unknown`.

In [ ]:
unseen = unseen_categories_by_fold(X, y)
display(unseen.pivot_table(index='atributo', values=['categorias_novas', 'linhas_afetadas'], aggfunc='sum'))

test_path = PROJECT_ROOT / 'data' / 'teste_sem_rotulo.csv'
if test_path.exists():
    test = pd.read_csv(test_path)
    for column in KNOWN_CATEGORICAL:
        if column in test.columns:
            train_levels = set(X[column].dropna())
            test_levels = set(test[column].astype('string').str.strip().str.lower().dropna())
            print(f'{column}: {len(test_levels - train_levels)} nível(is) do teste ausente(s) no treino')

## 4. Dimensionalidade gerada por técnica

In [ ]:
dims = encoded_dimensions(X, preparation_report)
display(dims)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(dims['variante'], dims['colunas_categoricas'], color='#4C72B0')
ax.set(xlabel='Colunas geradas pelos atributos categóricos', title='Dimensionalidade por codificação')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'codificacao_dimensionalidade.png', dpi=150)
plt.show()

## 5. Comparação por validação cruzada

In [ ]:
resultados = evaluate_encoding_strategies(X, y, preparation_report)
resultados.to_csv(RESULTS_DIR / 'resultados_codificacao.csv', index=False)
colunas = ['variante', 'f1_macro_media', 'f1_macro_desvio', 'precision_anomaly_media',
           'recall_anomaly_media', 'colunas_categoricas', 'tempo_ajuste_s_media']
display(resultados[colunas].round(4))

In [ ]:
# Regra de decisão definida antes de ver o resultado: entre as variantes a até 1 desvio do melhor
# F1-macro, escolher a de menor dimensionalidade (mais simples e mais estável para o teste).
melhor = resultados.iloc[0]
empatadas = resultados[resultados['f1_macro_media'] >= melhor['f1_macro_media'] - melhor['f1_macro_desvio']]
escolhida = empatadas.sort_values(['colunas_categoricas', 'f1_macro_media'], ascending=[True, False]).iloc[0]
print('Melhor F1-macro:', melhor['variante'], f"({melhor['f1_macro_media']:.4f} ± {melhor['f1_macro_desvio']:.4f})")
print('Escolhida pela regra:', escolhida['variante'])

## 6. Justificativa

- **One-hot** não impõe ordem entre categorias e, com `handle_unknown='ignore'`, uma categoria nova vira um
  vetor de zeros em vez de quebrar a previsão. O custo é a dimensionalidade: `service` sozinho responde pela
  maior parte das colunas geradas, e várias das suas categorias têm poucas linhas, o que deixa os coeficientes
  dessas colunas instáveis.
- **One-hot com raras agrupadas** (`min_frequency=0,1%`) mantém a ausência de ordem e reduz as colunas
  juntando os níveis raros de `service` num único nível “infrequente”, aprendido em cada dobra de treino.
- **Ordinal** gera uma coluna por atributo, mas inventa uma ordem alfabética sem significado. Para a regressão
  logística isso vira uma relação linear artificial; para árvores o efeito é menor. É mantida como controle
  negativo do que **não** fazer em modelos lineares.
- **Frequência** também gera uma coluna por atributo e é calculada só no treino da dobra (categoria nova
  vira 0), sem usar o rótulo, o que evita o vazamento do *target encoding*. A desvantagem é que categorias
  diferentes com a mesma frequência ficam indistinguíveis.
- **Híbrida** aplica one-hot onde a cardinalidade é baixa (`protocol_type`, `flag`) e frequência em `service`,
  atacando exatamente o problema de alta cardinalidade apontado no enunciado.
- **Imputação**: a categoria `ausente` só difere da moda quando existem categóricas faltantes (ver seção 2).
  Se a ausência for informativa (diferente entre as classes), `ausente` preserva esse sinal; com a moda, ele se
  perde.

**Decisão para a configuração comum:** a variante impressa na célula acima, pela regra definida antes de ver
os resultados. *(Atualizar com os números finais ao rodar com a base oficial.)*